# Notebook 02 -- word_list_coverage

**Asks:** which facets can the word list say anything about at all?

The word list built in notebook 01 (nine buckets of search patterns, one per facet)
is the only scorer in this pipeline that uses no language model. That is exactly
what makes it worth having: its agreement with the model ensemble is a more
independent check than the models' agreement with one another, which is why prereg
6.7 names it as safeguard 1.

But a check can only check what it can see. If the word list fires on one response
in a hundred, the per-facet agreement between it and the ensemble is computed on a
handful of positive cases and is noise. Worse, that silence reads as "no independent
contradiction" when what it really means is "no independent measurement at all". So
the design sets a floor in advance (design 6, registered as a deviation in design 0):

> A stream counts as independent corroboration for a facet only if it produces a
> usable, nonzero-variance signal on that facet -- operationally, fires on at least
> 5% of the relevant items.

This notebook measures, for every facet, the share of responses the word list fires
on, and records which facets clear that floor. Nothing here is a finding about the
participants. It is a finding about the instrument.

**Pooled and per item, and which of the two the pipeline reads.** Coverage is
measured twice: once over all three open-ended items together, and once separately
within each item. The per-item rows are the ones that matter downstream -- notebook
10 asks the coverage question facet by facet AT EACH ITEM, and notebook 11 asks it
for revenge at the backward item. A facet can clear the floor pooled and be empty at
the one item where its gate is run, so the pooled row is a summary for the reader
and never the verdict the chain acts on.

**What earlier runs showed, and why this is its own notebook.** On the first tranche
of 300 participants the word list fired on revenge in about 0.7% of responses,
suffering about 2.5%, proportionality about 1.0%, censure about 1.4% and moral
balance about 0.7%. Four of the nine facets are therefore expected to fail the floor
outright, with suffering borderline. Those numbers are the reason this step exists
on its own: a disqualified leg has to be a visible, checkable result rather than a
preregistered safeguard quietly dropped.

**Reads:** `facet_dictionary_scores.csv` (notebook 01)
**Writes:** `word_list_coverage.csv` -- one row per facet per slice, the pooled slice
plus one per item type.

## Config

In [ ]:
# === CONFIG: the only cell you edit ===
# Every switch in the pipeline lives here. Nothing below this cell needs changing.

RUN_MODE    = "full"        # "smoke" = first 20 rows, costs pennies, proves the
                            #           whole chain works end to end.
                            # "full"  = the real corpus.
HUMAN_GOLD  = False         # False = the pre-registered pipeline. Language models
                            #         do the coding, exactly as prereg 6.7 says.
                            #         This is the default and it runs start to
                            #         finish with no human step.
                            # True  = ALSO run the human-coder arm. That arm is a
                            #         DEVIATION from the registration and needs the
                            #         OSF addendum posted first.
WORKDIR     = "/content/retribution_nlp"   # where every file in the chain lives
RANDOM_SEED = 20260629      # frozen. Changing it re-draws every random split.

# --- PER-NOTEBOOK: the three lines below differ in each notebook ---
NB_ID       = "02"                     # this notebook's number
NB_NAME     = "word_list_coverage"     # this notebook's job, in two words
NB_ASKS     = "Ask which facets the word list can say anything about at all."

## Setup

In [ ]:
# === SETUP: folders, the manifest, and the two functions every notebook uses ===
# WHAT THIS CELL DOES: makes the working folder, loads the manifest, and defines
# need() and done(). You never edit this cell.
#
# THE MANIFEST is a small file called _manifest.json that lives beside the data.
# Every notebook records what it produced in there. The next notebook reads it to
# check its inputs exist. That is what lets a notebook tell you "run 03 first"
# instead of dying on a missing file three cells later.

import os, json, hashlib, datetime, sys

os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)
MANIFEST = os.path.join(WORKDIR, "_manifest.json")

def _load_manifest():
    """Read the manifest, or start an empty one on the first notebook."""
    if os.path.exists(MANIFEST):
        with open(MANIFEST) as f:
            return json.load(f)
    return {"files": {}, "runs": []}

def _sha256(path, cap=8_000_000):
    """First 8 MB is plenty to catch a file that changed underneath us."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        h.update(f.read(cap))
    return h.hexdigest()[:16]

class MissingInput(Exception):
    """Raised when a required upstream file is not there yet."""
    pass

def need(filename, produced_by=None, optional=False):
    """Return the path to an input file, or explain which notebook makes it.

    This is the function that makes the chain navigable. A missing input stops
    the notebook HERE, with a sentence telling you what to run, rather than
    surfacing as a confusing error further down.
    """
    path = os.path.join(WORKDIR, filename)
    if os.path.exists(path):
        rec = _load_manifest()["files"].get(filename, {})
        stamp = rec.get("written_at", "unknown time")
        print(f"  [ok]   {filename}  (written {stamp})")
        return path
    if optional:
        print(f"  [skip] {filename} is not here. This notebook will carry on "
              f"without it and say so in its output.")
        return None
    where = f" Run notebook {produced_by} first." if produced_by else ""
    raise MissingInput(
        f"\n\n  MISSING INPUT: {filename}\n"
        f"  This notebook cannot run without it.{where}\n"
        f"  Files present in {WORKDIR}:\n    "
        + "\n    ".join(sorted(os.listdir(WORKDIR)) or ["(none)"]) + "\n")

def done(filename, rows=None, note=""):
    """Record a file this notebook produced, so later notebooks can find it."""
    path = os.path.join(WORKDIR, filename)
    if not os.path.exists(path):
        raise FileNotFoundError(f"done() called for {filename} but it is not there")
    m = _load_manifest()
    m["files"][filename] = {
        "written_by": f"{NB_ID}_{NB_NAME}",
        "written_at": datetime.datetime.now().isoformat(timespec="seconds"),
        "rows": rows,
        "sha16": _sha256(path),
        "run_mode": RUN_MODE,
        "note": note,
    }
    m["runs"] = [r for r in m["runs"] if r.get("nb") != NB_ID]
    m["runs"].append({"nb": NB_ID, "name": NB_NAME, "run_mode": RUN_MODE,
                      "human_gold": HUMAN_GOLD,
                      "at": datetime.datetime.now().isoformat(timespec="seconds")})
    with open(MANIFEST, "w") as f:
        json.dump(m, f, indent=1)
    size = os.path.getsize(path)
    print(f"  [wrote] {filename}  ({rows if rows is not None else '?'} rows, "
          f"{size:,} bytes)")

print(f"NOTEBOOK {NB_ID} -- {NB_NAME}")
print(f"  {NB_ASKS}")
print(f"  mode: RUN_MODE={RUN_MODE}, HUMAN_GOLD={HUMAN_GOLD}")
print(f"  folder: {WORKDIR}")
_m = _load_manifest()
if _m["runs"]:
    print("  already run in this folder: "
          + ", ".join(sorted(r["nb"] for r in _m["runs"])))
else:
    print("  this folder is empty: notebook 00 is the place to start.")

NOTEBOOK 02 -- word_list_coverage
  Ask which facets the word list can say anything about at all.
  mode: RUN_MODE=full, HUMAN_GOLD=False
  folder: /content/retribution_nlp
  this folder is empty: notebook 00 is the place to start.


## Load

In [ ]:
# === LOAD (1 of 2): the scores from notebook 01, and the shape they must have ===
# WHAT THIS CELL DOES: reads facet_dictionary_scores.csv and checks the three things
# every number below rests on -- that the columns this notebook needs are present,
# that the file holds the three items this study asked, and that it holds exactly one
# row per participant per item.
# WHY THE CHECKS ARE HERE RATHER THAN ASSUMED: the corpus is 948 participants x 3
# open-ended answers = 2,844 rows, keyed on (response_id, item_type). Notebook 00
# asserts that key for openends_long.csv, at its cell "VALIDATE LOUDLY, PART 1".
# Notebook 01 turns those rows into this file, and this notebook checks the file it
# actually reads rather than trusting a guarantee made about a different file.
# WHAT BREAKS WITHOUT THEM: a coverage percentage is a count of responses divided by
# a count of rows. Duplicate rows and every denominator below grows, every percentage
# moves, and a facet can cross the 5% floor -- or drop under it -- with nothing
# printed to say so. That floor decides which facets the word list is allowed to
# corroborate for the rest of the chain, so a silent denominator here is a silent
# change to a preregistered safeguard.

import pandas as pd

DICT_IN = need("facet_dictionary_scores.csv", produced_by="01")
dict_df = pd.read_csv(DICT_IN)

# The columns without which nothing here can be computed. Checked by name so that a
# file missing one stops with a sentence naming the notebook to re-run, the way
# need() does for a missing file, instead of a bare KeyError several cells down.
REQUIRED_COLS = ["response_id", "item_type", "n_words"]
_absent = [c for c in REQUIRED_COLS if c not in dict_df.columns]
assert not _absent, (
    f"facet_dictionary_scores.csv has no {_absent} column. That file is written by "
    "notebook 01 from openends_long.csv; re-run notebook 01, and if it too is "
    "missing the column, re-run notebook 00.")

# The three items, spelled as notebook 00 writes them. Every consumer looks its
# coverage row up by these exact strings, so a relabelled item column would leave 27
# of the 36 output rows empty and explain them as "no non-blank responses in this
# slice" -- a false reason on a full corpus, and the word-list leg would then vanish
# from notebook 10's agreement matrix without a true word of explanation.
ITEM_TYPES = ["ranking", "backward", "forward"]
_seen = sorted(dict_df["item_type"].dropna().astype(str).unique())
assert _seen == sorted(ITEM_TYPES), (
    f"item_type holds {_seen}. This notebook, notebook 10 and notebook 11 all key on "
    f"exactly {sorted(ITEM_TYPES)}. Re-run notebooks 00 and 01.")

# ONE ROW PER PARTICIPANT PER ITEM, checked on the file as read.
assert not dict_df.duplicated(["response_id", "item_type"]).any(), (
    "facet_dictionary_scores.csv holds more than one row for some participant x "
    "item_type. Every percentage below would be computed on an inflated denominator, "
    "and the printed counts would not show it. Re-run notebook 01.")
N_PART = dict_df["response_id"].nunique()
assert len(dict_df) == N_PART * len(ITEM_TYPES), (
    f"{len(dict_df)} rows for {N_PART} participants across {len(ITEM_TYPES)} items: "
    "somebody is missing an item, so the per-item slices below would be drawn from "
    "different people. Re-run notebook 00, which builds the long file.")
print(f"  {len(dict_df)} rows = {N_PART} participants x {len(ITEM_TYPES)} items, "
      "one row per person per item.")
if RUN_MODE != "smoke" and len(dict_df) != 948 * 3:
    print(f"  [check] the design expects {948 * 3} rows (948 analytic participants "
          f"x 3) and this file has {len(dict_df)}. A run that skipped the prereg 6.2 "
          "exclusions gives 1015 x 3 = 3045; see notebook 00.")

  [ok]   facet_dictionary_scores.csv  (written unknown time)
  2844 rows = 948 participants x 3 items, one row per person per item.


In [ ]:
# === LOAD (2 of 2): which columns are facets, and which rows said something ===
# WHY BLANKS ARE MARKED: the coverage floor is a statement about responses that said
# something. A participant who left the box empty is no evidence that the word list
# missed anything. Leave the blanks in the denominator and every facet's coverage is
# deflated by the blank rate, which would push a borderline facet under the floor for
# a reason that has nothing to do with the word list.

# RUN_MODE = "smoke" is a plumbing check, not a measurement. Twenty participants
# cannot estimate a 1% prevalence, so it says so loudly here and writes "not run"
# into the verdict columns further down instead of a decision.
SMOKE_PARTICIPANTS = 20
if RUN_MODE == "smoke":
    # Trim by PARTICIPANT, never by row. Every person contributes three rows, one per
    # open-ended question, and notebooks 00, 03, 06 and 08 all cut on participants. A
    # head(20) here would keep 20 ROWS, six and two thirds of a person, and the three
    # per-item slices below would then be drawn from different participants: the
    # by-item table would compare different people instead of one person's three
    # answers, and the ranking column would rest on seven people and the forward
    # column on six.
    _keep = pd.unique(dict_df["response_id"])[:SMOKE_PARTICIPANTS]
    dict_df = dict_df[dict_df["response_id"].isin(_keep)].reset_index(drop=True)
    print(f"  SMOKE MODE: {len(_keep)} participants, {len(dict_df)} rows. Every "
          "percentage below comes from those rows and NOTHING is decided on them: "
          "clears_floor is written as 'not run' for every facet, so a plumbing check "
          "cannot disqualify a facet for the rest of the chain.")

# The facet names are recovered from the file rather than re-typed here. Notebook 01
# writes three columns per facet -- <facet>, <facet>_per100 and <facet>_negzeroed --
# so a bare column with both of its partners present is a facet, and the original
# column order is preserved. Re-typing the nine names in a second notebook is exactly
# how two lists silently drift apart after someone edits the dictionary. The next
# cell checks this discovered list against the scheme.
FACET_NAMES = [c for c in dict_df.columns
               if f"{c}_per100" in dict_df.columns
               and f"{c}_negzeroed" in dict_df.columns]

# n_words is the word count carried down from the extraction step; 0 marks a blank.
dict_df["nonblank"] = pd.to_numeric(dict_df["n_words"], errors="coerce").fillna(0) > 0

print(f"  non-blank responses: {int(dict_df['nonblank'].sum())} of {len(dict_df)}")
print(f"  facet columns found ({len(FACET_NAMES)}): {', '.join(FACET_NAMES)}")

  non-blank responses: 2837 of 2844
  facet columns found (9): desert, proportionality, censure, moral_balance, revenge, suffering, deterrence, incapacitation, rehabilitation


## Do

In [ ]:
# === THE RULE, AND THE FACETS IT APPLIES TO ===
# WHAT THIS CELL DOES: sets the coverage floor, records the class groupings, and
# checks that the file really carries the nine facets the scheme names. Nothing is
# measured here.

# The floor, from design 6 (registered as a deviation in design 0, the table of
# departures from the preregistration). A stream counts as independent corroboration
# for a facet only where it produces a usable, nonzero-variance signal on that facet
# -- operationally, where it fires on at least 5% of the relevant items.
# WHY A FLOOR AT ALL, AND WHY 5%: at 0.7% of roughly 300 backward responses the word
# list fires on two of them. A per-facet agreement computed on two positive cases is
# noise, and -- worse -- its silence reads as "no independent contradiction" when
# what it actually means is "no independent measurement". The floor makes that
# difference explicit instead of letting an empty leg pass for a clean one.
# FOR THE RECORD: adding this rule is a DEVIATION. Prereg 6.7 names the word list as
# a standing check with no prevalence condition attached. The rule is stated in
# advance, and the disqualified legs are still computed and reported with their
# counts, so the disqualification is auditable rather than assumed.
COVERAGE_FLOOR_PCT = 5.0

# The class groupings, carried from the dictionary. PRINCIPLED and VENGEFUL are the
# two sides of E4's co-occurrence question. CONSEQUENTIALIST is a third axis, scored
# so the retributive facets are measured against a full menu of punishment
# justifications rather than in a vacuum, and NEVER summed into any retributivism
# total.
FACET_CLASS = {
    "desert": "principled", "proportionality": "principled",
    "censure": "principled", "moral_balance": "principled",
    "revenge": "vengeful", "suffering": "vengeful",
    "deterrence": "consequentialist", "incapacitation": "consequentialist",
    "rehabilitation": "consequentialist",
}

# THE FILE MUST CARRY EVERY FACET IN THE SCHEME, AND NO OTHERS.
# WHAT THIS CATCHES: a facet is only discovered above when all three of its columns
# are present. Lose one partner column and the facet is not reported as "not run" --
# it ceases to exist in the coverage file, and notebook 11 then reports E4's
# preregistered word-list corroborator as not run because "word_list_coverage.csv is
# not here", a reason that never happened about a file sitting in the folder. Lose
# all of them (an older pipeline wrote <facet> with no partners) and the file written
# below would be a header with no rows, which raises EmptyDataError inside notebook
# 10. Both failures are silent at this end, which is why this is a stop.
_gone = [f for f in FACET_CLASS if f not in FACET_NAMES]
_extra = [f for f in FACET_NAMES if f not in FACET_CLASS]
assert not _gone and not _extra, (
    f"the scheme names {len(FACET_CLASS)} facets and this file yields "
    f"{len(FACET_NAMES)}. Missing here: {_gone or 'none'}. Unexpected here: "
    f"{_extra or 'none'}. A facet is counted only when <facet>, <facet>_per100 and "
    "<facet>_negzeroed are all present. Re-run notebook 01; if the dictionary really "
    "did change, change FACET_CLASS and FACET_STANDING with it.")

In [ ]:
# === WHAT EACH FACET IS FOR, so the verdict can say what is lost ===
# TWO STANDINGS, AND ONLY TWO. "Gate-eligible" means the facet has an admissible
# survey scale to be validated against; prereg 5.15 names exactly four such
# facet-to-scale mappings and no others. "Descriptive-only" means prevalence and
# reliability are reported for the facet and no facet-to-survey claim is made about
# it. Neither label makes any E3-E7 test confirmatory: the whole family is
# exploratory. What a facet is ABOUT -- principled, vengeful, consequentialist -- is
# a different question with a different answer, and it travels in the facet_class
# column beside this one.
FACET_STANDING = {
    "desert":          ("gate-eligible",
                        "anchored to the `deserved` scale; half of the E4 pair"),
    "revenge":         ("gate-eligible",
                        "anchored to the `revenge` scale; the other half of E4"),
    "suffering":       ("gate-eligible",
                        "anchored to the `suffering` scale"),
    "proportionality": ("descriptive-only",
                        "demoted on ~1% prevalence; its gate numbers are reported anyway"),
    "censure":         ("descriptive-only",
                        "no admissible survey anchor exists in this instrument"),
    "moral_balance":   ("descriptive-only",
                        "no admissible survey anchor exists in this instrument"),
    "deterrence":      ("descriptive-only",
                        "consequentialist convergent sanity check, never a gate target"),
    "incapacitation":  ("descriptive-only",
                        "consequentialist convergent sanity check, never a gate target"),
    "rehabilitation":  ("descriptive-only",
                        "consequentialist convergent sanity check, never a gate target"),
}

# Prevalence measured on the first tranche of 300 participants, printed beside the
# realized figures further down so that a large move is visible immediately. Every
# rule keyed to a base rate is re-evaluated on the realized corpus: these five
# numbers are the expectation, the table below is the decision.
FIRST_TRANCHE_PCT = {"revenge": 0.7, "suffering": 2.5, "proportionality": 1.0,
                     "censure": 1.4, "moral_balance": 0.7}

In [ ]:
# === HOW COVERAGE IS MEASURED ===
# Coverage = the share of responses on which the word list fired AT LEAST ONCE.
# Prevalence, not intensity: the floor is stated in terms of how many responses the
# instrument can see at all, not how loudly it fires on the ones it does see. That is
# the quantity design 6 states the rule in, and swapping in a mean count would answer
# a different question.
#
# THE RAW COUNTS ARE USED, NEVER THE NEGATION-ZEROED ONES. The negation whitelist
# zeroes a facet's count where a verified negation-plus-cue span is present ("don't
# want him to suffer"). Design 6 bars that adjusted column from every gate
# computation, and this coverage verdict IS a gate decision: it decides whether a
# facet's word-list leg is admissible at all. Score coverage off the zeroed column
# and the dictionary's explicitly disclaimed stance handling leaks into the gate
# through the back door. The zeroed figures are reported further down, side by side,
# as a diagnostic that decides nothing.

def wilson_ci(k, n, z=1.96):
    """95% confidence interval for a proportion, in the Wilson form.

    A confidence interval is the range of true rates consistent with what we saw. The
    Wilson form is used rather than the textbook one because at 0.7% on a few hundred
    rows the textbook interval runs below zero and is useless exactly where it is
    needed, which is at the facets sitting near the floor.
    """
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    denom  = 1 + z**2 / n
    centre = p + z**2 / (2 * n)
    half   = z * ((p * (1 - p) / n + z**2 / (4 * n**2)) ** 0.5)
    return (100 * max(0.0, (centre - half) / denom),
            100 * min(1.0, (centre + half) / denom))

In [ ]:
# === ONE COVERAGE RECORD PER FACET PER SLICE ===
def coverage_row(frame, facet, scope):
    """Coverage for one facet over one slice of the corpus, with its verdict."""
    sub = frame[frame["nonblank"]]
    n = len(sub)
    standing, purpose = FACET_STANDING.get(facet, ("unclassified", ""))
    # run_mode rides on every row so that a reader of the CSV, and not only a reader
    # of this screen, can tell a measurement from a plumbing check.
    rec = {"facet": facet, "facet_class": FACET_CLASS.get(facet, "unclassified"),
           "standing": standing, "purpose": purpose, "scope": scope,
           "n_responses": n, "run_mode": RUN_MODE}
    if n == 0:
        # NOT RUN is written out in words, never left as a blank cell. A blank in a
        # coverage table cannot be told apart from a failure, and the two mean
        # opposite things: one says the word list looked and found nothing, the other
        # says nobody looked.
        rec.update({"n_nonzero": 0, "pct_nonzero": None, "ci_lo_pct": None,
                    "ci_hi_pct": None, "clears_floor": "not run",
                    "near_floor_flag": "not run", "pct_nonzero_negzeroed": None,
                    "verdict": "not run -- no non-blank responses in this slice"})
        return rec
    # to_numeric with fillna(0) so a hand-edited file with a stray blank counts as a
    # no-hit rather than raising here.
    k = int((pd.to_numeric(sub[facet], errors="coerce").fillna(0) > 0).sum())
    pct = 100 * k / n
    lo, hi = wilson_ci(k, n)
    kz = int((pd.to_numeric(sub[facet + "_negzeroed"], errors="coerce")
              .fillna(0) > 0).sum())
    clears = "yes" if pct >= COVERAGE_FLOOR_PCT else "no"
    verdict = ("usable as an independent check for this facet" if clears == "yes"
               else "DISQUALIFIED as an independent check: below the "
                    f"{COVERAGE_FLOOR_PCT:g}% floor")
    # A near miss is flagged but decides nothing -- the rule is the point estimate
    # against the floor. The flag is here so that a facet whose interval touches the
    # floor (the design calls suffering borderline for exactly this reason) is read
    # with its interval instead of as a clean pass or a clean fail. A facet that fired
    # on NOTHING is not borderline: at zero hits the interval reaches the floor only
    # because the slice is small, which is a fact about the slice.
    near = "yes" if (k > 0 and lo <= COVERAGE_FLOOR_PCT <= hi) else "no"
    if RUN_MODE == "smoke":
        # A smoke run must not leave a verdict behind for the rest of the chain.
        # Notebooks 10 and 11 both CARRY this verdict rather than recompute it, and
        # neither of them reads RUN_MODE, so a decision written here on 20
        # participants would rewrite a preregistered safeguard for every notebook
        # after it. The percentages are still written, and they are still real for
        # these rows; what is withheld is the decision.
        clears = near = "not run"
        verdict = (f"not run -- RUN_MODE=smoke: {n} non-blank rows from "
                   f"{SMOKE_PARTICIPANTS} participants is a plumbing check, not a "
                   "prevalence estimate. Re-run with RUN_MODE='full' before any "
                   "notebook reads this file.")
    rec.update({"n_nonzero": k, "pct_nonzero": round(pct, 2),
                "ci_lo_pct": round(lo, 2), "ci_hi_pct": round(hi, 2),
                "clears_floor": clears, "near_floor_flag": near,
                "pct_nonzero_negzeroed": round(100 * kz / n, 2), "verdict": verdict})
    return rec

In [ ]:
# === THE HEADLINE TABLE: coverage over the whole corpus ===
# WHAT IT IS AND IS NOT. This is the pooled picture, every item together. It is the
# fastest read on which facets the word list can see at all, and it is NOT the number
# any later notebook acts on: the gate is applied item by item, and the per-item table
# two cells down is what notebooks 10 and 11 look up.
# HAZARD: edit the dictionary in notebook 01 and you change which facets clear this
# floor, which changes which convergence legs are admissible later. Re-read this
# table after any lexicon change.
all_rows = [coverage_row(dict_df, f, "all_items") for f in FACET_NAMES]

print("COVERAGE, ALL THREE ITEMS POOLED: % of non-blank responses where the word "
      "list fired at least once")
print(f"floor = {COVERAGE_FLOOR_PCT:.0f}%  (design 6)\n")
print(f"{'facet':16s} {'class':16s} {'%':>6s} {'95% CI':>14s} {'hits':>6s}   verdict")
print("-" * 92)
for r in all_rows:
    if r["pct_nonzero"] is None:
        print(f"{r['facet']:16s} {r['facet_class']:16s} {'not run':>6s}")
        continue
    ci = f"[{r['ci_lo_pct']:.1f}, {r['ci_hi_pct']:.1f}]"
    mark = {"yes": "PASS", "no": "FAIL <-- not an independent leg",
            "not run": "not run (smoke mode decides nothing)"}[r["clears_floor"]]
    if r["near_floor_flag"] == "yes":
        note = "  (interval touches the floor)"
    elif r["n_nonzero"] == 0 and r["ci_hi_pct"] >= COVERAGE_FLOOR_PCT:
        note = "  (fired on nothing here; too few rows to place it)"
    else:
        note = ""
    print(f"{r['facet']:16s} {r['facet_class']:16s} {r['pct_nonzero']:6.1f} "
          f"{ci:>14s} {r['n_nonzero']:6d}   {mark}{note}")
print(f"\n  hits = responses with >=1 pattern firing, out of "
      f"{int(dict_df['nonblank'].sum())} non-blank responses.")

COVERAGE, ALL THREE ITEMS POOLED: % of non-blank responses where the word list fired at least once
floor = 5%  (design 6)

facet            class                 %         95% CI   hits   verdict
--------------------------------------------------------------------------------------------
desert           principled         20.1   [18.7, 21.6]    570   PASS
proportionality  principled          1.2     [0.9, 1.7]     34   FAIL <-- not an independent leg
censure          principled          0.9     [0.6, 1.3]     26   FAIL <-- not an independent leg
moral_balance    principled          0.5     [0.3, 0.8]     14   FAIL <-- not an independent leg
revenge          vengeful            0.6     [0.3, 0.9]     16   FAIL <-- not an independent leg
suffering        vengeful            1.8     [1.4, 2.4]     52   FAIL <-- not an independent leg
deterrence       consequentialist   15.3   [14.0, 16.7]    434   PASS
incapacitation   consequentialist   13.8   [12.6, 15.2]    393   PASS
rehabilitation  

In [ ]:
# === EXPECTED VERSUS REALIZED ===
# The first-tranche figures sit beside the realized ones so a big move is visible. A
# facet that was 0.7% in the first 300 and is 6% here has either had its lexicon
# expanded or is meeting a different corpus, and either way the leg-validity decision
# downstream moved with it. These are the five facets the design flagged in advance
# as the ones at risk of failing the floor.
print("\nFirst tranche (n=300) versus this corpus:")
print(f"{'facet':16s} {'first tranche':>14s} {'realized':>12s}")
for r in all_rows:
    exp = FIRST_TRANCHE_PCT.get(r["facet"])
    if exp is None:
        continue
    got = "not run" if r["pct_nonzero"] is None else f"{r['pct_nonzero']:.1f}%"
    print(f"{r['facet']:16s} {exp:13.1f}% {got:>12s}")


First tranche (n=300) versus this corpus:
facet             first tranche     realized
proportionality            1.0%         1.2%
censure                    1.4%         0.9%
moral_balance              0.7%         0.5%
revenge                    0.7%         0.6%
suffering                  2.5%         1.8%


In [ ]:
# === COVERAGE AT THE SITE WHERE THE GATE IS ACTUALLY RUN ===
# WHY A SECOND TABLE, AND WHY IT IS THE ONE THAT COUNTS: the floor is stated over
# "the relevant items", and which items are relevant depends on the facet. The
# facet-versus-survey gate is evaluated on the BACKWARD item for desert, revenge and
# suffering, because that item is the cleanest register match to the closed-ended
# scales (design 3). A facet can clear 5% pooled across all three items and still be
# empty at the one site where its gate is run, and the pooled number would hide that.
# Notebook 10 reads a coverage row per facet PER ITEM and notebook 11 reads revenge at
# the backward item; neither reads the pooled row.
# READ THE COLUMN LENGTHS ALONGSIDE: median words run about 32 for ranking, 19 for
# backward and 13 for forward, so the shortest item gives any pattern the least text
# to fire in and will tend to show the lowest coverage on every facet. A low forward
# number is partly a fact about the item, not only about the facet.
# THE COUNTS ARE PRINTED BESIDE THE PERCENTAGES because design 6 requires a
# disqualified leg to be reported WITH ITS COUNTS: "0.2" and "2 of 948 responses" are
# the same fact, but only the second one tells a reader how little is behind it.
GATE_ITEM = {"desert": "backward", "revenge": "backward", "suffering": "backward"}

item_rows = []
for it in ITEM_TYPES:
    sl = dict_df[dict_df["item_type"] == it]
    for f in FACET_NAMES:
        item_rows.append(coverage_row(sl, f, it))
by_key = {(r["facet"], r["scope"]): r for r in item_rows}

print("\nCOVERAGE BY ITEM -- % of that item's non-blank responses with >=1 hit, and "
      "the hit count behind it:")
print(f"{'facet':16s}" + "".join(f"{it:>16s}" for it in ITEM_TYPES))
for f in FACET_NAMES:
    cells = ""
    for it in ITEM_TYPES:
        r = by_key[(f, it)]
        if r["pct_nonzero"] is None:
            cells += f"{'not run':>16s}"
        else:
            star = "*" if r["clears_floor"] == "yes" else " "
            cells += f"{r['pct_nonzero']:9.1f} ({r['n_nonzero']:3d}){star}"
    print(f"{f:16s}{cells}")
if RUN_MODE == "smoke":
    print("  No star appears anywhere: a smoke run reaches no verdict, so no facet "
          "clears anything.")
else:
    print(f"  * clears the {COVERAGE_FLOOR_PCT:.0f}% floor at that item.")
print("  'not run' = no non-blank responses there.")
print("  The gate for desert, revenge and suffering is run on the BACKWARD column.")


COVERAGE BY ITEM -- % of that item's non-blank responses with >=1 hit, and the hit count behind it:
facet                    ranking        backward         forward
desert               31.9 (301)*     18.7 (177)*      9.7 ( 92)*
proportionality       0.9 (  9)       0.9 (  9)       1.7 ( 16) 
censure               0.6 (  6)       0.3 (  3)       1.8 ( 17) 
moral_balance         0.4 (  4)       0.4 (  4)       0.6 (  6) 
revenge               0.8 (  8)       0.2 (  2)       0.6 (  6) 
suffering             0.4 (  4)       2.2 ( 21)       2.9 ( 27) 
deterrence           37.1 (351)*      2.3 ( 22)       6.5 ( 61)*
incapacitation       22.4 (212)*     13.0 (123)*      6.1 ( 58)*
rehabilitation       53.0 (501)*      3.6 ( 34)      10.5 ( 99)*
  * clears the 5% floor at that item.
  'not run' = no non-blank responses there.
  The gate for desert, revenge and suffering is run on the BACKWARD column.


In [ ]:
# === WHAT THIS MEANS, IN WORDS (1 of 2): where the leg exists and where it does not ===
# This is the sentence the rest of the pipeline depends on, and it is a PER-ITEM
# sentence because that is how the pipeline asks the question. A facet listed as
# unusable at an item is not "measured as absent" there by the word list. It is NOT
# MEASURED. Where a later notebook shows a blank word-list agreement for it, that
# blank is silence: neither corroboration nor contradiction.
usable = [r["facet"] for r in all_rows if r["clears_floor"] == "yes"]
lost   = [r["facet"] for r in all_rows if r["clears_floor"] == "no"]
notrun = [r for r in all_rows if r["clears_floor"] == "not run"]

if RUN_MODE == "smoke":
    print("\nNo verdict was reached: this is a smoke run. Every clears_floor cell in "
          "the file below reads 'not run', so nothing here disqualifies or admits a "
          "facet anywhere downstream. Re-run with RUN_MODE='full' to decide.")
else:
    print("\nWHERE THE WORD LIST CAN SERVE AS AN INDEPENDENT CHECK, ITEM BY ITEM:")
    for f in FACET_NAMES:
        ok = [it for it in ITEM_TYPES if by_key[(f, it)]["clears_floor"] == "yes"]
        nr = [it for it in ITEM_TYPES if by_key[(f, it)]["clears_floor"] == "not run"]
        if len(nr) == len(ITEM_TYPES):
            # Nothing was measured anywhere, which is not the same as measured and
            # found wanting at every item.
            print(f"  {f:16s} not measured at any item (no non-blank responses)")
            continue
        line = f"  {f:16s} usable at: {', '.join(ok) if ok else 'NO ITEM'}"
        if nr:
            line += f"   (not run at: {', '.join(nr)})"
        print(line)
    print("\nAt the item where each gate-eligible facet's gate is run (design 3):")
    for f, it in GATE_ITEM.items():
        r = by_key.get((f, it))
        if r is None:
            continue
        verdict = {"yes": "usable", "no": "NOT usable"}.get(r["clears_floor"], "not run")
        pct = "--" if r["pct_nonzero"] is None else f"{r['pct_nonzero']:.1f}%"
        print(f"  {f:16s} {it:9s} {pct:>7s} of {r['n_responses']} responses "
              f"({r['n_nonzero']} hits) -- {verdict}")
    # The trap this table exists to close: a facet that looks usable pooled and is
    # not usable where its leg is actually wanted.
    gap = [(f, it) for f in FACET_NAMES for it in ITEM_TYPES
           if f in usable and by_key[(f, it)]["clears_floor"] == "no"]
    if gap:
        print("\n  READ THIS BEFORE QUOTING THE POOLED LIST. These facets clear the "
              "floor pooled and\n  fail it at the item named, where they are "
              "therefore not an independent leg:")
        for f, it in gap:
            print(f"    {f} at {it}: {by_key[(f, it)]['pct_nonzero']:.1f}% "
                  f"({by_key[(f, it)]['n_nonzero']} hits)")


WHERE THE WORD LIST CAN SERVE AS AN INDEPENDENT CHECK, ITEM BY ITEM:
  desert           usable at: ranking, backward, forward
  proportionality  usable at: NO ITEM
  censure          usable at: NO ITEM
  moral_balance    usable at: NO ITEM
  revenge          usable at: NO ITEM
  suffering        usable at: NO ITEM
  deterrence       usable at: ranking, forward
  incapacitation   usable at: ranking, backward, forward
  rehabilitation   usable at: ranking, forward

At the item where each gate-eligible facet's gate is run (design 3):
  desert           backward    18.7% of 947 responses (177 hits) -- usable
  revenge          backward     0.2% of 947 responses (2 hits) -- NOT usable
  suffering        backward     2.2% of 947 responses (21 hits) -- NOT usable

  READ THIS BEFORE QUOTING THE POOLED LIST. These facets clear the floor pooled and
  fail it at the item named, where they are therefore not an independent leg:
    deterrence at backward: 2.3% (22 hits)
    rehabilitation at back

In [ ]:
# === WHAT THIS MEANS, IN WORDS (2 of 2): the pooled summary, and what is left ===
# The pooled list is printed after the per-item one, and second, deliberately: it is
# a summary for the reader and no notebook in this chain acts on it.
if RUN_MODE != "smoke":
    print("\nPooled over all three items (a summary; no later notebook reads this "
          "row):")
    print("  can check: " + (", ".join(usable) if usable else "(none)"))
    print("  cannot:    " + (", ".join(lost) if lost else "(none)"))
    for r in all_rows:
        if r["clears_floor"] == "no":
            print(f"  - {r['facet']}: fires on {r['n_nonzero']} of "
                  f"{r['n_responses']} responses ({r['pct_nonzero']:.1f}%); "
                  f"{r['purpose']}.")
# Never a bare "not run": each row carries the reason it was not run, and the reason
# is printed, because "nobody looked" and "we looked and found nothing" are the two
# readings this notebook exists to keep apart. Facets are grouped by reason so that a
# whole-run reason (a smoke run) prints once rather than nine times.
_why = {}
for r in notrun:
    _why.setdefault(r["verdict"], []).append(r["facet"])
for why, facets in _why.items():
    print(f"  - {', '.join(facets)}:\n      {why}")

# What is left for a facet the word list cannot check. Design 6 names the human coder
# as the strongest independent check that remains -- and human coding is optional in
# this pipeline and off by default (HUMAN_GOLD), because prereg 6.7 puts the model
# ensemble in place of human raters. Where it is off, these facets rest on the two
# registered gate clauses alone: agreement across models, and correlation with the
# matching survey scale. Censure and moral balance have no admissible survey scale
# either, so for those two only the model-agreement clause can ever run.
print("\nWhere the word-list leg is gone, design 6 names the human coder as the")
print("strongest independent check that remains; that arm is optional and currently")
print(f"HUMAN_GOLD={HUMAN_GOLD}. With it off, those facets rest on model agreement")
print("plus the survey correlation, and censure and moral balance have no admissible")
print("survey scale, so for them only model agreement applies.")


Pooled over all three items (a summary; no later notebook reads this row):
  can check: desert, deterrence, incapacitation, rehabilitation
  cannot:    proportionality, censure, moral_balance, revenge, suffering
  - proportionality: fires on 34 of 2837 responses (1.2%); demoted on ~1% prevalence; its gate numbers are reported anyway.
  - censure: fires on 26 of 2837 responses (0.9%); no admissible survey anchor exists in this instrument.
  - moral_balance: fires on 14 of 2837 responses (0.5%); no admissible survey anchor exists in this instrument.
  - revenge: fires on 16 of 2837 responses (0.6%); anchored to the `revenge` scale; the other half of E4.
  - suffering: fires on 52 of 2837 responses (1.8%); anchored to the `suffering` scale.

Where the word-list leg is gone, design 6 names the human coder as the
strongest independent check that remains; that arm is optional and currently
HUMAN_GOLD=False. With it off, those facets rest on model agreement
plus the survey correlation, and c

In [ ]:
# === RAW VERSUS NEGATION-ZEROED, SIDE BY SIDE (diagnostic, decides nothing) ===
# Design 6 requires that the raw and negation-zeroed counts be reported together, and
# that the zeroed column be kept out of every gate computation. Every verdict above
# is raw. This table exists so the size of the whitelist's effect is visible: if
# zeroing moves a facet's coverage by more than a rounding error, the whitelist is
# doing more work than a hand-curated list of individually inspected phrases should
# be doing, and it needs re-reading before anything downstream is trusted.
# On a run where the whitelist is still empty the two columns are identical, which is
# correct rather than a sign that the cell did not run.
print("\nRaw versus negation-zeroed coverage (diagnostic only, feeds no gate):")
print(f"{'facet':16s} {'raw %':>8s} {'zeroed %':>10s} {'shift':>8s}")
for r in all_rows:
    if r["pct_nonzero"] is None:
        print(f"{r['facet']:16s} {'not run':>8s}")
        continue
    shift = r["pct_nonzero_negzeroed"] - r["pct_nonzero"]
    note = "   <-- inspect the whitelist" if abs(shift) > 0.5 else ""
    print(f"{r['facet']:16s} {r['pct_nonzero']:8.1f} "
          f"{r['pct_nonzero_negzeroed']:10.1f} {shift:8.1f}{note}")


Raw versus negation-zeroed coverage (diagnostic only, feeds no gate):
facet               raw %   zeroed %    shift
desert               20.1       20.1      0.0
proportionality       1.2        1.2      0.0
censure               0.9        0.9      0.0
moral_balance         0.5        0.5      0.0
revenge               0.6        0.6      0.0
suffering             1.8        1.8      0.0
deterrence           15.3       15.3      0.0
incapacitation       13.8       13.8      0.0
rehabilitation       22.4       22.4      0.0


## Save and check

In [ ]:
# === SAVE: one row per facet per slice ===
# The file carries the pooled row (scope = all_items) AND one row per item_type, so a
# later notebook can ask the coverage question at the exact site it is scoring rather
# than re-deriving it from the pooled number.
# clears_floor and near_floor_flag are written as the words "yes" / "no" / "not run"
# rather than as true/false, so that a slice nobody measured can never be read as a
# slice that failed. That distinction is the reason this notebook exists.
coverage_df = pd.DataFrame(all_rows + item_rows)
COLS = ["facet", "facet_class", "standing", "scope", "run_mode", "n_responses",
        "n_nonzero", "pct_nonzero", "ci_lo_pct", "ci_hi_pct", "clears_floor",
        "near_floor_flag", "pct_nonzero_negzeroed", "verdict", "purpose"]
coverage_df = coverage_df[[c for c in COLS if c in coverage_df.columns]]

# ONE ROW PER FACET PER SCOPE, checked before the file is handed on. This notebook
# merges nothing, so there is no fan-out to guard; what it must guard is the key of
# the file it writes. Notebook 10 selects coverage[(facet == f) & (scope == item)] and
# takes .iloc[0], and notebook 11 does the same for revenge at the backward item, so a
# duplicated pair would hand them whichever copy sorted first, silently, and a second
# row saying the opposite would never be seen.
assert not coverage_df.duplicated(["facet", "scope"]).any(), (
    "word_list_coverage.csv would carry two rows for the same facet x scope. Check "
    f"ITEM_TYPES ({ITEM_TYPES}) and the facet columns for a repeated entry.")

COV_OUT = "word_list_coverage.csv"
coverage_df.to_csv(COV_OUT, index=False)
done(COV_OUT, rows=len(coverage_df),
     note=(f"floor={COVERAGE_FLOOR_PCT}% of non-blank responses; raw counts only, "
           f"never the negation-zeroed ones; scope=all_items plus one row per item"
           + ("; SMOKE RUN, every clears_floor cell reads 'not run'"
              if RUN_MODE == "smoke" else "")))

  [wrote] word_list_coverage.csv  (36 rows, 7,263 bytes)


In [ ]:
# === WHAT THIS RUN DECIDED ===
print("\n" + "=" * 72)
print(f"NOTEBOOK {NB_ID} DONE -- {NB_NAME}")
print(f"  {NB_ASKS}")
if RUN_MODE == "smoke":
    print("  SMOKE RUN: NOTHING WAS DECIDED. Every clears_floor cell in")
    print("  word_list_coverage.csv reads 'not run', which is what keeps a")
    print(f"  {SMOKE_PARTICIPANTS}-participant plumbing check from disqualifying a")
    print("  preregistered safeguard for the rest of the chain. Re-run this notebook")
    print("  with RUN_MODE='full' before notebooks 10 and 11 read the file.")
else:
    print(f"  pooled over all three items, {len(usable)} of {len(FACET_NAMES)} "
          f"facets clear the floor: {', '.join(usable) if usable else 'none'}")
    print("  item by item, which is the verdict notebooks 10 and 11 actually read:")
    for it in ITEM_TYPES:
        ok = [f for f in FACET_NAMES if by_key[(f, it)]["clears_floor"] == "yes"]
        print(f"    {it:9s} {len(ok)} of {len(FACET_NAMES)}: "
              f"{', '.join(ok) if ok else 'none'}")
    print("  Disqualified legs are computed and reported above with their counts,")
    print("  which is what makes the disqualification checkable rather than assumed.")
print("  next: notebook 03.")


NOTEBOOK 02 DONE -- word_list_coverage
  Ask which facets the word list can say anything about at all.
  pooled over all three items, 4 of 9 facets clear the floor: desert, deterrence, incapacitation, rehabilitation
  item by item, which is the verdict notebooks 10 and 11 actually read:
    ranking   4 of 9: desert, deterrence, incapacitation, rehabilitation
    backward  2 of 9: desert, incapacitation
    forward   4 of 9: desert, deterrence, incapacitation, rehabilitation
  Disqualified legs are computed and reported above with their counts,
  which is what makes the disqualification checkable rather than assumed.
  next: notebook 03.
